<h2>Heart_Disease</h2>

Problem Statement:

You are the data scientist at a medical research facility. The facility wants you to
build a machine learning model to classify if the given data of a patient should tell
if the patient is at the risk of a heart attack.

Tasks To Be Performed:

1. Data Analysis:
a. Import the dataset
b. Get information about the dataset (mean, max, min, quartiles etc.)
c. Find the correlation between all fields

2. Data Visualization:
a. Visualize the number of patients having a heart disease and not having
a heart disease
b. Visualize the age and whether a patient has disease or not
c. Visualize correlation between all features using a heat map

3. Logistic Regression:
a. Build a simple logistic regression model:
i. Divide the dataset in 70:30 ratio
ii. Build the model on train set and predict the values on test set
iii. Build the confusion matrix and get the accuracy score

4. Decision Tree:
a. Build a decision tree model:
i. Divide the dataset in 70:30 ratio
ii. Build the model on train set and predict the values on test set
iii. Build the confusion matrix and calculate the accuracy
iv. Visualize the decision tree using the Graphviz package

5. Random Forest:
a. Build a Random Forest model:
i. Divide the dataset in 70:30 ratio
ii. Build the model on train set and predict the values on test set
iii. Build the confusion matrix and calculate the accuracy
iv. Visualize the model using the Graphviz package

6. Select the best model
a. Print the confusion matrix of all classifiers
b. Print the classification report of all classifiers
c. Calculate Recall Precision and F1 score of all the models
d. Visualize confusion matrix using heatmaps
e. Select the best model based on the best accuracies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
heart_disease = pd.read_csv("C:/Users/AHSAN RAZA/Desktop/dataset.csv")
print(heart_disease)

In [ ]:
heart_disease.shape

In [ ]:
heart_disease.columns

In [ ]:
heart_disease.info()

In [ ]:
heart_disease.describe()

In [ ]:
corr_matrix = heart_disease.corr()
print(corr_matrix)

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(x="target", data=heart_disease)
plt.xlabel("Heart Disease (0 = No, 1 = Yes)")
plt.ylabel("Number of Patients")
plt.title("Number of Patients With and Without Heart Disease")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x="target", y="age", data=heart_disease)
plt.xlabel("Heart Disease (0 = No, 1 = Yes)")
plt.ylabel("Age")
plt.title("Age Distribution by Heart Disease Status")
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Splitting the data

x = heart_disease.drop(columns=['target'])
y = heart_disease['target']

from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression

lo = LogisticRegression(max_iter=1000)
lo.fit(x_train,y_train)

y_pred = lo.predict(x_test)

from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test,y_pred)
acc_score = accuracy_score(y_test,y_pred)

print('Confusion matrix:\n',cm)
print('Accuracy Score:\n',acc_score)

In [ ]:
!pip install graphviz

In [ ]:
# Decision Tree

from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier(random_state=42,max_depth=3)
dtc.fit(x_train,y_train)

y_pred = dtc.predict(x_test)

from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test,y_pred)
acc_score = accuracy_score(y_test,y_pred)

print('Confusion matrix:\n',cm)
print('Accuracy Score:\n',acc_score)

from sklearn.tree import export_graphviz 
import graphviz

dot_data = export_graphviz(
    dtc,
    out_file=None,
    feature_names=x.columns,
    class_names=["No Disease", "Disease"],
    filled=True,
    rounded=True,
)

graph = graphviz.Source(dot_data)
graph

In [ ]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf.fit(x_train, y_train)

y_pred = rf.predict(x_test)

cm = confusion_matrix(y_test,y_pred)
acc_score = accuracy_score(y_test,y_pred)

print('Confusion matrix:\n',cm)
print('Accuracy Score:\n',acc_score)

from sklearn.tree import export_graphviz
import graphviz

estimator = rf.estimators_[0]

dot_data = export_graphviz(
    estimator,
    out_file=None,
    feature_names=x.columns,
    class_names=["No Disease", "Disease"],
    filled=True,
    rounded=True
)

graph = graphviz.Source(dot_data)
graph

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": lo,  
    "Decision Tree": dtc,               
    "Random Forest": rf                 
}

metrics_list = []

for name, model in models.items():
    y_pred = model.predict(x_test)
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n{name} Confusion Matrix:\n", cm)
    
    # Classification Report
    print(f"\n{name} Classification Report:\n", classification_report(y_test, y_pred))
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Store metrics in a list
    metrics_list.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })
    
    # Heatmap for confusion matrix
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Disease","Disease"],
                yticklabels=["No Disease","Disease"])
    plt.title(f"{name} Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

metrics_df = pd.DataFrame(metrics_list)

print("\nAll Model Metrics:\n")
print(metrics_df)

best_model = metrics_df.loc[metrics_df["Accuracy"].idxmax(), "Model"]
best_accuracy = metrics_df["Accuracy"].max()
print(f"\nBest Model: {best_model} with Accuracy = {best_accuracy:.2f}")